# NLP Spelling Correction System

# Introduction:
This notebook presents an NLP-based spelling correction system that detects both non-word errors (misspelled words) and real-word errors (valid words used incorrectly in context). It combines edit distance, confusion pairs, bigram probabilities, and semantic similarity with an interactive Gradio interface.

### Setup & installs

Install required libraries (gradio, wordfreq, spacy) and download the English language model (en_core_web_sm) for text processing.

In [1]:
!pip -q install gradio wordfreq spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 125.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Imports & globals

Import necessary libraries, load the spaCy model, and set global variables to manage the system state.

In [2]:
import re
import math
import requests
from collections import Counter, defaultdict
from wordfreq import top_n_list, zipf_frequency
import spacy
import gradio as gry # Changed gr to gry

# Load spaCy for semantic similarity
nlp = spacy.load("en_core_web_sm")

# Global variables for system state
SYSTEM_INITIALIZED = False
FREQ = None
VOCAB = None
BG = None
V = 0
TOTAL = 0

### Confusion pairs & reliable words

Define common confusion pairs (like their/there or affect/effect) for real-word error detection, and list reliable words to skip unnecessary checks.

In [3]:
# Expanded list of common confusion pairs
COMMON_CONFUSION_PAIRS = {
    "accept": "except", "except": "accept",
    "affect": "effect", "effect": "affect",
    "advice": "advise", "advise": "advice",
    "ensure": "insure", "insure": "ensure",
    "their": "there", "there": "their", "they're": "their",
    "then": "than", "than": "then",
    "your": "you're", "you're": "your",
    "its": "it's", "it's": "its",
    "to": "too", "too": "to", "two": "to",
    "loose": "lose", "lose": "loose",
    "weather": "whether", "whether": "weather",
    "which": "witch", "witch": "which",
    "here": "hear", "hear": "here",
    "our": "are", "are": "our",
    'of': 'off', 'off': 'of',
    'for': 'four', 'four': 'for',
    'who': 'whom', 'whom': 'who',
    'whose': "who's", "who's": 'whose',
    'where': 'were', 'were': 'where', "we're": 'where', 'wear': 'where',
    'right': 'write', 'write': 'right', 'rite': 'right',
    'site': 'sight', 'sight': 'site', 'cite': 'site',
    'piece': 'peace', 'peace': 'piece',
    'quiet': 'quite', 'quite': 'quiet',
}


# Reliable words (skip analysis for these)
RELIABLE_WORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
    'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'it', 'we', 'they',
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had',
    'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'can',
    'not', 'no', 'yes', 'all', 'any', 'some', 'many', 'much', 'more', 'most', 'very'
}

### System initialization (download corpus, build vocab/bigrams)

Initialize the system by downloading the corpus, creating vocabulary and frequency counts, and building bigrams for context checking.

In [4]:
def initialize_system():
    global SYSTEM_INITIALIZED, FREQ, VOCAB, BG, V, TOTAL
    if SYSTEM_INITIALIZED:
        return "✅ System already initialized!"
    try:
        CORPUS_URL = "https://www.gutenberg.org/files/3300/3300-0.txt"  # Wealth of Nations
        text = requests.get(CORPUS_URL).text.lower()
        tokens = re.findall(r"[a-z]+", text)
        FREQ = Counter(tokens)
        VOCAB = set(tokens)
        VOCAB |= set(top_n_list('en', n=100_000))
        BG = defaultdict(int)
        for a, b in zip(tokens, tokens[1:]):
            BG[(a, b)] += 1
        V = max(1, len(VOCAB))
        TOTAL = sum(FREQ.values())
        SYSTEM_INITIALIZED = True
        return f"✅ System initialized successfully!\n📊 Vocabulary: {len(VOCAB):,} words\n📚 Corpus: {TOTAL:,} tokens\n🔗 Bigrams: {len(BG):,} pairs"
    except Exception as e:
        return f"❌ Initialization failed: {str(e)}"

### Edit distance & candidate generation

Generate candidate corrections using edit distance (insertions, deletions, substitutions, transpositions) and rank them by closeness to the input word.

In [5]:
letters = "abcdefghijklmnopqrstuvwxyz"

def edits1(w):
    splits = [(w[:i], w[i:]) for i in range(len(w)+1)]
    deletes = [L+R[1:] for L, R in splits if R]
    transposes = [L+R[1]+R[0]+R[2:] for L, R in splits if len(R)>1]
    replaces = [L+c+R[1:] for L, R in splits if R for c in letters]
    inserts = [L+c+R for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def edits2(w, cap=4000):
    out = set()
    for e1 in edits1(w):
        for e2 in edits1(e1):
            if e2 in VOCAB:
                out.add(e2)
                if len(out) >= cap:
                    return out
    return out

def lev(a, b):
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j]+1, curr[j-1]+1, prev[j-1]+(ca!=cb)))
        prev = curr
    return prev[-1]

def suggest_top3(word, want=3):
    if not SYSTEM_INITIALIZED or not VOCAB:
        return []
    w = word.lower()
    if w in VOCAB:
        return []
    cand = {c for c in edits1(w) if c in VOCAB}
    if len(cand) < want:
        cand.update(edits2(w, cap=2000))
    cand = {c for c in cand if abs(len(c)-len(w)) <= 3}
    if len(cand) < want:
        extra_cand = edits2(w, cap=3000)
        cand.update({c for c in extra_cand if c in VOCAB})
        cand = {c for c in cand if abs(len(c)-len(w)) <= 4}
    ranked = sorted(
        cand,
        key=lambda c: (
            lev(w, c),
            abs(len(w)-len(c)),
            0 if (w and c and w[0]==c[0]) else 1,
            0 if (w and c and w[-1]==c[-1]) else 1,
            -zipf_frequency(c, 'en'),
            -FREQ.get(c, 0),
            c
        )
    )
    out = []
    for s in ranked:
        if s != w and s not in out:
            out.append(s)
        if len(out) >= want:
            break
    if len(out) < 2 and len(ranked) >= 2:
        for s in ranked[:10]:
            if s != w and s not in out:
                out.append(s)
            if len(out) >= 2:
                break
    return out[:want]

### Semantic similarity helper

Use spaCy to measure semantic similarity between two words, helping detect context-based errors.

In [6]:
def semantic_similarity(word1, word2):
    w1 = nlp(word1)
    w2 = nlp(word2)
    return w1.similarity(w2)

### Language model: unigram/bigram log-probabilities

Calculate unigram and bigram log-probabilities to evaluate how likely a word or word pair is in context.

In [7]:
def logP_bigram(w1, w2, alpha=1.0):
    if not SYSTEM_INITIALIZED:
        return 0.0
    if not w1 or not w2:
        num = FREQ.get(w2, 0) + alpha
        den = max(1, TOTAL + alpha*V)
        return math.log(num/den)
    num = BG.get((w1, w2), 0) + alpha
    den = max(1, FREQ.get(w1, 0) + alpha*V)
    return math.log(num/den)

### Real-word error detection (context + semantics)

Detect real-word errors by checking confusion pairs, context with bigrams, and semantic similarity to suggest better replacements.

In [8]:
def detect_real_word_errors(text, words, lower):
    real_word_errors = []
    for i, w in enumerate(lower):
        if w not in VOCAB or w in RELIABLE_WORDS:
            continue
        prev = lower[i-1] if i>0 else None
        nxt = lower[i+1] if i+1<len(lower) else None

        # Common confusion pairs
        if w in COMMON_CONFUSION_PAIRS:
            suggestion = COMMON_CONFUSION_PAIRS[w]
            real_word_errors.append({
                'word': words[i],
                'suggestion': suggestion,
                'position': i,
                'reason': "Common confusion pair"
            })
            continue

        # Base probability
        base = (logP_bigram(prev, w) if prev else 0) + (logP_bigram(w, nxt) if nxt else 0)

        # Candidate suggestions
        cands = suggest_top3(w) or list({c for c in edits1(w) if c in VOCAB})[:3]
        best = None
        best_gain = 0.0
        for c in cands:
            if c == w:
                continue
            score = (logP_bigram(prev, c) if prev else 0) + (logP_bigram(c, nxt) if nxt else 0)
            gain = score - base
            # tie-break with semantic similarity
            if gain > best_gain or (gain == best_gain and semantic_similarity(w, c) > 0.7):
                best_gain, best = gain, c

        if best and best_gain > math.log(1.05) and lev(w, best) <= 2 and best != w:
            real_word_errors.append({
                'word': words[i],
                'suggestion': best,
                'position': i,
                'reason': "Contextual and semantic analysis"
            })
    return real_word_errors

### Check Spelling (Non-word + Real-word)

Main function to check spelling by detecting both non-word and real-word errors, then generating suggestions and reports.

In [9]:
def check_spelling(text):
    if not SYSTEM_INITIALIZED:
        return "❌ Please initialize the system first.", "", ""
    if not text.strip():
        return "📝 Please enter some text to check.", "", ""
    words = re.findall(r"[A-Za-z']+", text)[:5000]
    lower = [w.lower() for w in words]

    non_word_errors = []
    real_word_errors = detect_real_word_errors(text, words, lower)

    for i, w in enumerate(lower):
        if w not in VOCAB:
            sugs = suggest_top3(w)
            non_word_errors.append({
                'word': words[i],
                'suggestions': sugs,
                'position': i
            })

    if not non_word_errors and not real_word_errors:
        return "✅ No errors detected!", "✅ No non-word errors found!", "✅ No real-word errors found!"

    highlighted_text = create_highlighted_text(text, words, non_word_errors, real_word_errors)
    non_word_report = create_simple_non_word_report(non_word_errors)
    real_word_report = create_simple_real_word_report(real_word_errors)
    return highlighted_text, non_word_report, real_word_report

### Reporting & Highlighting Helpers

Create reports for detected errors and highlight them in the text for clear visualization.

In [10]:
def create_simple_non_word_report(non_word_errors):
    if not non_word_errors:
        return "✅ **No Non-word Errors Found!**"
    report = f"🔴 **NON-WORD ERRORS** ({len(non_word_errors)} found)\n\n"
    for i, error in enumerate(non_word_errors, 1):
        word = error['word']
        suggestions = error['suggestions']
        sugs_text = ', '.join(suggestions) if suggestions else 'no suggestions'
        report += f"**{i}. {word}**\n"
        report += f"   💡 Suggestions: {sugs_text}\n\n"
    return report

def create_simple_real_word_report(real_word_errors):
    if not real_word_errors:
        return "✅ **No Real-word Errors Found!**"
    report = f"🟡 **REAL-WORD ERRORS** ({len(real_word_errors)} found)\n\n"
    for i, error in enumerate(real_word_errors, 1):
        word = error['word']
        suggestion = error['suggestion']
        reason = error['reason']
        report += f"**{i}. {word}**\n"
        report += f"   💡 Maybe: {suggestion}\n"
        report += f"   📌 Reason: {reason}\n\n"
    return report

def create_highlighted_text(text, words, non_word_errors, real_word_errors):
    error_positions = {}
    for error in non_word_errors:
        error_positions[error['position']] = {
            'type': 'non-word',
            'word': error['word'],
            'suggestions': error['suggestions']
        }
    for error in real_word_errors:
        error_positions[error['position']] = {
            'type': 'real-word',
            'word': error['word'],
            'suggestion': error['suggestion']
        }

    result_parts = []
    current_pos = 0
    for i, word in enumerate(words):
        word_start = text.lower().find(word.lower(), current_pos)
        if word_start == -1:
            continue
        if word_start > current_pos:
            result_parts.append(text[current_pos:word_start])
        if i in error_positions:
            error_info = error_positions[i]
            if error_info['type'] == 'non-word':
                highlighted_word = (
                    f'<mark style="background-color: #ffcccb; color: #d32f2f; font-weight: bold;">{word}</mark>'
                )
            else:
                highlighted_word = (
                    f'<mark style="background-color: #fff3cd; color: #f57c00; font-weight: bold;">{word}</mark>'
                )
            result_parts.append(highlighted_word)
        else:
            result_parts.append(word)
        current_pos = word_start + len(word)

    if current_pos < len(text):
        result_parts.append(text[current_pos:])

    highlighted = ''.join(result_parts)
    legend = '''
    <div style="margin-top: 10px; padding: 8px; background-color: #f5f5f5; border-radius: 5px; font-size: 12px;">
    <strong>Legend:</strong>
    <span style="background-color: #ffcccb; color: #d32f2f; padding: 2px 4px; border-radius: 2px;">🔴 Non-word Error</span>
    <span style="background-color: #fff3cd; color: #f57c00; padding: 2px 4px; border-radius: 2px; margin-left: 8px;">🟡 Real-word Error</span>
    </div>'''
    return f'<div style="font-family: Arial; line-height: 1.6; padding: 10px; border: 1px solid #ddd; border-radius: 5px;">{highlighted}</div>{legend}'

### Utilities: clear text, vocabulary search, word analysis

Utility functions to clear text, search vocabulary, and analyze words with frequency, context, and suggestions.

In [11]:
def clear_text():
    return "", "", ""

def search_vocabulary(query=""):
    if not SYSTEM_INITIALIZED or not VOCAB:
        return "❌ Please initialize the system first."
    if not query.strip():
        if FREQ:
            most_common = FREQ.most_common(20)
            result = "📊 Most Frequent Words:\n\n"
            for word, count in most_common:
                result += f"• {word} ({count:,} occurrences)\n"
            return result
        return "📝 Enter a search term or leave empty to see most frequent words."
    query = query.lower()
    matches = [w for w in VOCAB if query in w][:50]
    if not matches:
        return f"❌ No words found containing '{query}'"
    result = f"🔍 Found {len(matches)} word(s) containing '{query}':\n\n"
    for word in sorted(matches)[:20]:
        freq = FREQ.get(word, 0)
        result += f"• {word}" + (f" ({freq} occurrences)" if freq > 0 else " (general vocabulary)") + "\n"
    if len(matches) > 20:
        result += f"\n... and {len(matches) - 20} more"
    return result

def analyze_word(word):
    if not SYSTEM_INITIALIZED or not VOCAB:
        return "❌ Please initialize the system first."
    if not word.strip():
        return "📝 Please enter a word to analyze."
    w = word.lower().strip()
    result = f"🔍 Analysis for '{word}':\n\n"
    result += f"📝 Word: {w}\n"
    result += f"📏 Length: {len(w)} characters\n"
    in_vocab = w in VOCAB
    result += f"📚 In vocabulary: {'✅ Yes' if in_vocab else '❌ No'}\n"
    if in_vocab:
        freq = FREQ.get(w, 0)
        result += f"📊 Corpus frequency: {freq:,} occurrences\n"
        result += f"🌍 Global frequency (Zipf): {zipf_frequency(w, 'en'):.2f}\n"
        if BG:
            following = [(w2, count) for (w1, w2), count in BG.items() if w1 == w]
            if following:
                following = sorted(following, key=lambda x: x[1], reverse=True)[:5]
                result += f"\n➡️ Commonly followed by:\n"
                for next_word, count in following:
                    result += f"   • {next_word} ({count} times)\n"
            preceding = [(w1, count) for (w1, w2), count in BG.items() if w2 == w]
            if preceding:
                preceding = sorted(preceding, key=lambda x: x[1], reverse=True)[:5]
                result += f"\n⬅️ Commonly preceded by:\n"
                for prev_word, count in preceding:
                    result += f"   • {prev_word} ({count} times)\n"
    else:
        suggestions = suggest_top3(w)
        if suggestions:
            result += f"\n💡 Suggestions:\n"
            for suggestion in suggestions:
                result += f"   • {suggestion}\n"
    return result

### Gradio UI (Blocks layout)

Build the Gradio user interface with text input, error detection results, vocabulary search, and word analysis features.

In [12]:
def create_interface():
    with gry.Blocks(
        theme=gry.themes.Soft(),
        title="Advanced NLP Spelling Correction System",
        css="""
        .header-text {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            font-size: 28px;
            font-weight: bold;
            margin-bottom: 10px;
        }
        .subtitle { color: #666; font-size: 16px; margin-bottom: 5px; }
        .features { color: #888; font-size: 14px; margin-bottom: 20px; }
        .init-btn {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
            border: none !important; color: white !important; font-weight: bold !important;
            padding: 12px 24px !important; border-radius: 8px !important; margin-bottom: 20px !important;
        }
        .section-header {
            font-size: 20px; font-weight: bold; color: #333; margin: 20px 0 10px 0;
            display: flex; align-items: center; gap: 8px;
        }
        """
    ) as demo:
        gry.HTML("""
        <div style="text-align: center; margin-bottom: 30px;">
            <div style="display: flex; align-items: center; justify-content: center; gap: 10px; margin-bottom: 10px;">
                <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 8px 12px; border-radius: 8px; font-weight: bold;">abc</div>
                <h1 class="header-text">NLP Spelling Correction System</h1>
            </div>
            <p class="subtitle">Built with Edit Distance, Bigram Analysis & Probabilistic Modeling</p>
            <p class="features"><strong>Features:</strong> Non-word & Real-word error detection • Context-aware suggestions • 100K+ word vocabulary</p>
        </div>
        """)

        with gry.Row():
            init_btn = gry.Button("🚀 Initialize System", elem_classes="init-btn", size="lg")
            init_output = gry.Textbox(label="", visible=False)
        init_btn.click(initialize_system, outputs=init_output).then(
            lambda x: gry.Textbox(value=x, visible=True),
            inputs=init_output, outputs=init_output
        )

        with gry.Row():
            with gry.Column(scale=1):
                gry.HTML('<div class="section-header">📝 Text Editor</div>')
                gry.Markdown("**Text to Check**\nType or paste text to check for spelling errors")
                text_input = gry.Textbox(
                    placeholder="Enter your text here (max 500 characters)...",
                    lines=8, max_lines=12, show_label=False
                )
                with gry.Row():
                    check_btn = gry.Button("✅ Check Spelling", variant="primary", scale=2)
                    clear_btn = gry.Button("🗑️ Clear Text", scale=1)
            with gry.Column(scale=1):
                gry.HTML('<div class="section-header">🎯 Highlighted Text</div>')
                results_output = gry.HTML(label="", show_label=False)

        with gry.Row():
            with gry.Column(scale=1):
                gry.HTML('<div class="section-header">🔴 Non-word Errors</div>')
                non_word_output = gry.Markdown(value="*Non-word error analysis will appear here*", show_label=False)
            with gry.Column(scale=1):
                gry.HTML('<div class="section-header">🟡 Real-word Errors</div>')
                real_word_output = gry.Markdown(value="*Real-word error analysis will appear here*", show_label=False)

        with gry.Row():
            with gry.Column(scale=1):
                gry.HTML('<div class="section-header">📚 Vocabulary Browser</div>')
                vocab_input = gry.Textbox(label="Search Words", placeholder="Search vocabulary...", info="Leave empty to see most frequent words")
                vocab_output = gry.Textbox(lines=6, show_label=False, interactive=False)
            with gry.Column(scale=1):
                gry.HTML('<div class="section-header">🔍 Word Analysis</div>')
                analyze_input = gry.Textbox(label="Word to Analyze", placeholder="like", info="Get detailed statistics and bigram information")
                analyze_output = gry.Textbox(lines=6, show_label=False, interactive=False)

        check_btn.click(check_spelling, inputs=text_input, outputs=[results_output, non_word_output, real_word_output])
        clear_btn.click(clear_text, outputs=[text_input, results_output, non_word_output, real_word_output])
        vocab_input.submit(search_vocabulary, inputs=vocab_input, outputs=vocab_output)
        analyze_input.submit(analyze_word, inputs=analyze_input, outputs=analyze_output)
    return demo

### Launch app

Launch the Gradio app to run the spelling correction system with an interactive interface.

In [ ]:
if __name__ == "__main__":
    demo = create_interface()
    demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a506c0ed279f0a8a2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipython-input-2603920806.py:4: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  return w1.similarity(w2)
/tmp/ipython-input-2603920806.py:4: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models inste